# FIGNN Final  Validation

## 0. Environment
Sanity checks

In [1]:
import sys, importlib.util
print(sys.executable)
req=["numpy","pandas","scipy","sklearn","matplotlib","torch","torch_geometric"]
missing=[x for x in req if importlib.util.find_spec(x) is None]
print("Missing:",missing)
if missing: print("STOP: install/use the project fignn_env before continuing.")

d:\Portfolio Projects\QML EEG Paper\fignn_cni__project\fignn_env\Scripts\python.exe
Missing: []


## 1. Paths and authoritative files

In [28]:
from pathlib import Path
import os, json, copy, itertools, warnings
from scipy.io import loadmat

import numpy as np
import pandas as pd

ROOT = Path(r"D:\Portfolio Projects\QML EEG Paper\fignn_cni__project")

OUT = ROOT / "data" / "outputs"
PROC = ROOT / "data" / "data" / "processed"

S1 = OUT / "evaluation_loso_all29"
S2 = OUT / "evaluation_stage2_loso29"
LOSO = OUT / "loso"

TH = OUT / "plv_threshold_sensitivity"
EXP = OUT / "final_non_gpu_validation"
EXP.mkdir(parents=True, exist_ok=True)

GRAPH = PROC / "all_graphs_29sub.pkl"

SUBJECTS = list(range(1, 30))

S1CSV = S1 / "loso_all29_subject_metrics.csv"
S2CSV = S2 / "stage2_loso29_subject_metrics.csv"

# =============================================================
# Dataset metadata / provenance files
# =============================================================

RAW = ROOT / "data" / "data" / "raw"

NOTEBOOK_MAT = ROOT / "data" / "data" / "raw" / "notebook.mat"

# Trigger-list file: set this to the actual filename once placed
TRIGGER_LIST = ROOT / "data" / "data" / "raw" / "trigger_list.txt"

METADATA_OUT = EXP / "dataset_metadata"
METADATA_OUT.mkdir(parents=True, exist_ok=True)

print("RAW           :", RAW)
print("NOTEBOOK MAT  :", NOTEBOOK_MAT)
print("TRIGGER LIST  :", TRIGGER_LIST)
print("METADATA OUT  :", METADATA_OUT)

print("ROOT :", ROOT)
print("GRAPH:", GRAPH)
print("EXP  :", EXP)

assert S1CSV.exists(), f"Missing: {S1CSV}"
assert S2CSV.exists(), f"Missing: {S2CSV}"
assert GRAPH.exists(), f"Missing: {GRAPH}"

print("All required files found.")

RAW           : D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\data\raw
NOTEBOOK MAT  : D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\data\raw\notebook.mat
TRIGGER LIST  : D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\data\raw\trigger_list.txt
METADATA OUT  : D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\outputs\final_non_gpu_validation\dataset_metadata
ROOT : D:\Portfolio Projects\QML EEG Paper\fignn_cni__project
GRAPH: D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\data\processed\all_graphs_29sub.pkl
EXP  : D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\outputs\final_non_gpu_validation
All required files found.


## 2. Load and audit current Stage-1/Stage-2 results

In [20]:
stage1=pd.read_csv(S1CSV).sort_values("subject").reset_index(drop=True)
stage2=pd.read_csv(S2CSV)

need=["subject","d01","d02","d12","d01_d02","path_efficiency","relative_1back_position","perpendicular_distance","oneback_between","procrustes_similarity","direction_cosine"]
assert not [c for c in need if c not in stage1.columns]
stage1["primary_pass"]=(stage1.relative_1back_position>=0)&(stage1.relative_1back_position<=1)
stage1["distance_order_pass"]=stage1.d01<stage1.d02

print("Primary",stage1.primary_pass.sum(),"/29")
print("d01<d02",stage1.distance_order_pass.sum(),"/29")
print("Stage2 columns:",stage2.columns.tolist())

display(stage1.head())
display(stage2.head())

Primary 23 /29
d01<d02 27 /29
Stage2 columns: ['subject', 'procrustes_similarity', 'spearman_rho', 'spearman_p', 'monotonic', 'd01', 'd02', 'd12', 'position_0back', 'position_1back', 'position_2back']


,subject,d01,d02,d12,d01_d02,d12_d02,angle_deg,path_efficiency,relative_1back_position,perpendicular_distance,centrality,oneback_between,procrustes_similarity,direction_cosine,n_train_trajectories,checkpoint,primary_pass,distance_order_pass
0,1,0.012123,0.021680,0.014933,0.559199,0.688780,73.957670,0.801296,0.419143,0.008025,0.838286,True,0.903420,-0.834839,28,/lp-dev/24BCE1793/outputs/loso/subject_01/fign...,True,True
1,2,0.024682,0.088463,0.077595,0.279016,0.877146,71.814190,0.864931,0.154232,0.020569,0.308465,True,0.962859,0.875531,28,/lp-dev/24BCE1793/outputs/loso/subject_02/fign...,True,True
2,3,0.017334,0.044216,0.033566,0.392028,0.759139,63.021250,0.868684,0.288697,0.011727,0.577395,True,0.965416,0.854275,28,/lp-dev/24BCE1793/outputs/loso/subject_03/fign...,True,True
3,4,0.043441,0.067354,0.028198,0.644963,0.418652,40.807743,0.940190,0.620354,0.011885,0.759292,True,0.809539,0.954822,28,/lp-dev/24BCE1793/outputs/loso/subject_04/fign...,True,True
4,5,0.032191,0.059927,0.033896,0.537167,0.565616,49.884975,0.906797,0.484313,0.013924,0.968627,True,0.912726,0.942994,28,/lp-dev/24BCE1793/outputs/loso/subject_05/fign...,True,True


,subject,procrustes_similarity,spearman_rho,spearman_p,monotonic,d01,d02,d12,position_0back,position_1back,position_2back
0,1,0.743220,0.177183,1.587841e-07,True,0.744418,0.012222,0.622532,0.796323,1.011168,1.154282
1,2,0.832842,0.383912,1.001761e-31,True,0.436197,0.075025,1.236057,1.116781,1.153905,1.689677
2,3,0.829298,0.163560,1.342913e-06,True,0.674193,0.026549,0.576390,1.066949,1.271790,1.411831
3,4,0.743939,0.319065,6.747242e-22,True,0.680980,0.028899,0.430563,0.764629,1.111964,1.390994
4,5,0.899522,0.151266,7.985097e-06,True,0.684313,0.016537,0.640976,0.746838,0.911126,1.083887


## 3. Exact six-labeling rho permutation null
This uses only the authoritative pairwise distances.

In [4]:
def t_from_d(dAB, dAC, dBC):
    return (
        (dAB * dAB + dAC * dAC - dBC * dBC)
        / (2 * dAC * dAC)
        if dAC > 1e-15
        else np.nan
    )


rows = []

for _, r in stage1.iterrows():

    D = {
        (0, 1): r.d01,
        (0, 2): r.d02,
        (1, 2): r.d12
    }

    def d(a, b):
        return D[tuple(sorted((a, b)))]

    for p in itertools.permutations([0, 1, 2]):
        A, B, C = p

        rho = t_from_d(
            d(A, B),
            d(A, C),
            d(B, C)
        )

        rows.append({
            "subject": int(r.subject),
            "permutation": "-".join(map(str, p)),
            "rho": rho,
            "pass": 0 <= rho <= 1
        })


perm = pd.DataFrame(rows)

null_counts = (
    perm.groupby("permutation")["pass"]
        .sum()
        .reset_index(name="n_pass")
)

print(
    "Observed:",
    int(stage1.primary_pass.sum()),
    "/29"
)

display(null_counts)

print(
    "Global six-permutation p:",
    np.mean(
        null_counts.n_pass >= int(stage1.primary_pass.sum())
    )
)

perm.to_csv(
    EXP / "rho_exact_permutations.csv",
    index=False
)

null_counts.to_csv(
    EXP / "rho_permutation_counts.csv",
    index=False
)

Observed: 23 /29


,permutation,n_pass
0,0-1-2,23
1,0-2-1,4
2,1-0-2,8
3,1-2-0,4
4,2-0-1,8
5,2-1-0,23


Global six-permutation p: 0.3333333333333333


## 4. Load graphs + exact final FIGNN model

In [6]:
import pickle
import torch

with open(GRAPH, "rb") as f:
    all_graphs = pickle.load(f)

print(type(all_graphs), len(all_graphs))

g0 = all_graphs[0]

print(g0)
print("x:", g0.x.shape)
print("edge:", g0.edge_index.shape)
print("edge_attr:", g0.edge_attr.shape)

def attr(g, names):
    for n in names:
        if hasattr(g, n):
            return getattr(g, n)
    return None


def subject(g):
    value = attr(g, ["subject", "subject_id", "subj"])
    return int(torch.as_tensor(value).view(-1)[0])


def label(g):
    value = attr(g, ["y", "label"])
    return int(torch.as_tensor(value).view(-1)[0])


ss = np.array([subject(g) for g in all_graphs])
yy = np.array([label(g) for g in all_graphs])

print("Subjects:", np.unique(ss))
print("Labels:", np.unique(yy))

print("\nGraphs per subject:")
print(pd.Series(ss).value_counts().sort_index())

print("\nGraphs per label:")
print(pd.Series(yy).value_counts().sort_index())

<class 'list'> 27666
Data(x=[62, 4], edge_index=[2, 936], edge_attr=[936, 4], y=[1], subject=1, session=1)
x: torch.Size([62, 4])
edge: torch.Size([2, 936])
edge_attr: torch.Size([936, 4])
Subjects: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29]
Labels: [0 1 2 3]

Graphs per subject:
1     954
2     954
3     954
4     954
5     954
6     954
7     954
8     954
9     954
10    954
11    954
12    954
13    954
14    954
15    954
16    954
17    954
18    954
19    954
20    954
21    954
22    954
23    954
24    954
25    954
26    954
27    954
28    954
29    954
Name: count, dtype: int64

Graphs per label:
0    8352
1    8352
2    8352
3    2610
Name: count, dtype: int64


## 5. Frozen fold inference helpers

In [7]:
def norm_stats(s):
    p = LOSO / f"subject_{s:02d}" / "normalisation_stats.pt"
    assert p.exists(), p

    o = torch.load(
        p,
        map_location="cpu",
        weights_only=False
    )

    if isinstance(o, dict):
        mean = o.get("mean", o.get("x_mean", o.get("node_mean")))
        std = o.get("std", o.get("x_std", o.get("node_std")))
    else:
        mean, std = o[:2]

    return (
        torch.as_tensor(mean).float(),
        torch.as_tensor(std).float()
    )


def norm_graphs(gs, mean, std):
    out = []

    for g in gs:
        h = copy.deepcopy(g)
        h.x = (h.x.float() - mean) / std.clamp_min(1e-8)
        out.append(h)

    return out


def load_model(s):
    p = LOSO / f"subject_{s:02d}" / "fignn_best.pth"
    assert p.exists(), p

    m = FIGNNEncoder(
        node_in_dim=4,
        edge_dim=4,
        hidden_dim=128,
        latent_dim=64,
        n_heads=8,
        dropout=0.3
    )

    ck = torch.load(
        p,
        map_location="cpu",
        weights_only=False
    )

    m.load_state_dict(
        ck.get("model_state_dict", ck)
    )

    m.eval()
    return m


@torch.no_grad()
def encode(m, gs):
    zs = []
    ys = []
    ss = []

    for b in DataLoader(
        gs,
        batch_size=BATCH,
        shuffle=False,
        num_workers=0
    ):
        z = m(b)

        zs.append(z.cpu())
        ys.append(b.y.view(-1).cpu())
        ss.append(b.subject.view(-1).cpu())

    return (
        torch.cat(zs).numpy(),
        torch.cat(ys).numpy(),
        torch.cat(ss).numpy()
    )


print("helpers ready")

helpers ready


## 6. Cache 29 frozen-fold embeddings

In [8]:
EMB = EXP / "fold_embeddings"
EMB.mkdir(exist_ok=True)


def cache_path(s):
    return EMB / f"fold_{s:02d}.npz"


def run_fold(s):
    p = cache_path(s)

    if p.exists():
        a = np.load(p)
        return a["Z"], a["Y"], a["S"]

    # Outer LOSO split
    tr = [g for g in all_graphs if subject(g) != s]
    te = [g for g in all_graphs if subject(g) == s]

    # Fold-specific normalization
    mean, std = norm_stats(s)

    tr = norm_graphs(tr, mean, std)
    te = norm_graphs(te, mean, std)

    # Fold-specific frozen model
    m = load_model(s)

    # Encode training and held-out subjects
    z1, y1, s1 = encode(m, tr)
    z2, y2, s2 = encode(m, te)

    Z = np.r_[z1, z2]
    Y = np.r_[y1, y2]
    S = np.r_[s1, s2]

    np.savez_compressed(
        p,
        Z=Z,
        Y=Y,
        S=S
    )

    return Z, Y, S


for i, s in enumerate(SUBJECTS, 1):

    if cache_path(s).exists():
        print(f"[{i}/29] cached {s}")
    else:
        print(f"[{i}/29] encoding {s}")
        run_fold(s)


print(
    "caches:",
    len(list(EMB.glob("fold_*.npz")))
)

[1/29] cached 1
[2/29] cached 2
[3/29] cached 3
[4/29] cached 4
[5/29] cached 5
[6/29] cached 6
[7/29] cached 7
[8/29] cached 8
[9/29] cached 9
[10/29] cached 10
[11/29] cached 11
[12/29] cached 12
[13/29] cached 13
[14/29] cached 14
[15/29] cached 15
[16/29] cached 16
[17/29] cached 17
[18/29] cached 18
[19/29] cached 19
[20/29] cached 20
[21/29] cached 21
[22/29] cached 22
[23/29] cached 23
[24/29] cached 24
[25/29] cached 25
[26/29] cached 26
[27/29] cached 27
[28/29] cached 28
[29/29] cached 29
caches: 29


## 7. Verify regenerated geometry against the authoritative CSV

In [9]:
def cents(Z, Y):
    return np.stack([
        Z[Y == c].mean(0)
        for c in [0, 1, 2]
    ])


def rho(C):
    v = C[2] - C[0]

    return float(
        np.dot(C[1] - C[0], v)
        / np.dot(v, v)
    )


check = []

for s in SUBJECTS:

    a = np.load(cache_path(s))

    Z = a["Z"]
    Y = a["Y"]
    S = a["S"]

    C = cents(
        Z[S == s],
        Y[S == s]
    )

    rr = rho(C)

    csv_rho = stage1.loc[
        stage1.subject == s,
        "relative_1back_position"
    ].iloc[0]

    check.append([
        s,
        csv_rho,
        rr
    ])


check = pd.DataFrame(
    check,
    columns=[
        "subject",
        "csv_rho",
        "recomputed_rho"
    ]
)

check["abs_diff"] = abs(
    check["csv_rho"] - check["recomputed_rho"]
)

display(check)

print(
    "max rho diff:",
    check["abs_diff"].max()
)

assert check["abs_diff"].max() < 1e-4

print("Rho verification PASSED.")

,subject,csv_rho,recomputed_rho,abs_diff
0,1,0.419143,0.419142,8.635560e-07
1,2,0.154232,0.154232,1.670825e-07
2,3,0.288697,0.288697,3.550887e-07
3,4,0.620354,0.620354,2.951723e-07
4,5,0.484313,0.484313,4.186329e-07
5,6,0.469133,0.469133,1.823318e-07
6,7,0.299064,0.299064,1.802113e-07
7,8,0.427003,0.427004,3.608690e-07
8,9,0.002998,0.002997,2.286166e-07
9,10,0.120020,0.120020,2.905453e-07


max rho diff: 1.4589976501527824e-06
Rho verification PASSED.


## 8. Bootstrap rho CIs

In [10]:
rng = np.random.default_rng(20261002)
NBOOT = 1000


def boot_rho(Z, Y):
    A = [
        Z[Y == c]
        for c in [0, 1, 2]
    ]

    vals = np.empty(NBOOT)

    for i in range(NBOOT):
        C = [
            a[
                rng.integers(0, len(a), len(a))
            ].mean(0)
            for a in A
        ]

        vals[i] = rho(np.stack(C))

    return vals


br = []

for s in SUBJECTS:

    a = np.load(cache_path(s))

    z = a["Z"]
    y = a["Y"]
    ss = a["S"]

    v = boot_rho(
        z[ss == s],
        y[ss == s]
    )

    br.append([
        s,
        rho(cents(z[ss == s], y[ss == s])),
        np.percentile(v, 2.5),
        np.percentile(v, 97.5)
    ])


br = pd.DataFrame(
    br,
    columns=[
        "subject",
        "rho",
        "ci_low",
        "ci_high"
    ]
)

br["ci_inside_01"] = (
    (br["ci_low"] >= 0) &
    (br["ci_high"] <= 1)
)

display(br)

print(
    "CI entirely inside [0,1]:",
    br["ci_inside_01"].sum(),
    "/29"
)

br.to_csv(
    EXP / "rho_bootstrap_95CI.csv",
    index=False
)

,subject,rho,ci_low,ci_high,ci_inside_01
0,1,0.419142,-0.195996,1.124911,False
1,2,0.154232,0.033262,0.266840,True
2,3,0.288697,0.044359,0.544432,True
3,4,0.620354,0.453402,0.763586,True
4,5,0.484313,0.173194,0.774748,True
5,6,0.469133,0.237013,0.700579,True
6,7,0.299064,-0.043107,0.620547,False
7,8,0.427004,0.202859,0.641396,True
8,9,0.002997,-0.222959,0.179728,False
9,10,0.120020,-0.080302,0.311104,False


CI entirely inside [0,1]: 14 /29


## 9. Within-class dispersion + silhouette + frozen kNN

In [22]:
from sklearn.metrics import silhouette_score
from sklearn.neighbors import KNeighborsClassifier

disp = []
sil = []
knn = []

for s in SUBJECTS:

    a = np.load(cache_path(s))

    Z = a["Z"]
    Y = a["Y"]
    S = a["S"]

    tr = S != s
    te = S == s

    zt = Z[te]
    yt = Y[te]

    task = np.isin(yt, [0, 1, 2])

    zt_task = zt[task]
    yt_task = yt[task]

    sil_value = silhouette_score(
        zt_task,
        yt_task
    )

    

    # Within-class dispersion
    ds = {"subject": s}

    for c in [0, 1, 2]:

        zc = zt[yt == c]
        C = zc.mean(0)

        ds[f"dispersion_{c}"] = np.mean(
            np.linalg.norm(zc - C, axis=1)
        )

    disp.append(ds)

    sil.append([
        s,
        sil_value
    ])

    # Frozen 5-NN classifier
    clf = KNeighborsClassifier(
        n_neighbors=5,
        weights="distance"
    )

    tr_task = tr & np.isin(Y, [0, 1, 2])
    te_task = te & np.isin(Y, [0, 1, 2])

    clf.fit(Z[tr_task], Y[tr_task])

    acc = np.mean(
        clf.predict(Z[te_task]) == Y[te_task]
    )

    knn.append([
        s,
        np.mean(clf.predict(zt) == yt)
    ])


disp = pd.DataFrame(disp)

sil = pd.DataFrame(
    sil,
    columns=["subject", "silhouette"]
)

knn = pd.DataFrame(
    knn,
    columns=["subject", "knn_accuracy"]
)


display(disp)

print(
    "silhouette:",
    sil["silhouette"].mean(),
    sil["silhouette"].std(ddof=1),
    "positive:",
    (sil["silhouette"] > 0).sum()
)

print(
    "kNN:",
    knn["knn_accuracy"].mean(),
    knn["knn_accuracy"].std(ddof=1)
)


disp.to_csv(
    EXP / "within_class_dispersion.csv",
    index=False
)

sil.to_csv(
    EXP / "silhouette_loso.csv",
    index=False
)

knn.to_csv(
    EXP / "frozen_knn_loso.csv",
    index=False
)

,subject,dispersion_0,dispersion_1,dispersion_2
0,1,0.133437,0.141372,0.127464
1,2,0.094286,0.082217,0.069499
2,3,0.113978,0.107707,0.111115
3,4,0.104818,0.101526,0.092013
4,5,0.153193,0.148225,0.139006
5,6,0.100138,0.113088,0.104567
6,7,0.108619,0.112257,0.106341
7,8,0.105357,0.098638,0.085080
8,9,0.091491,0.087492,0.088729
9,10,0.113530,0.116300,0.117871


silhouette: -0.01457047969472177 0.011510064675205075 positive: 2
kNN: 0.3837923805392901 0.15411777347925182


## 10. Stage-2 centroids + translation + Ridge

In [14]:
from sklearn.linear_model import Ridge


def subj_cents(Z, Y, S, s):
    m = S == s

    return {
        name: Z[m & (Y == c)].mean(0)
        for c, name in [
            (0, "C0"),
            (1, "C1"),
            (2, "C2"),
            (3, "R")
        ]
        if np.any(m & (Y == c))
    }


rows = []

for s in SUBJECTS:

    a = np.load(cache_path(s))

    Z = a["Z"]
    Y = a["Y"]
    S = a["S"]

    cs = subj_cents(Z, Y, S, s)

    tr_subjects = [
        r for r in SUBJECTS
        if r != s
    ]

    X = []
    T = []

    for r in tr_subjects:

        cr = subj_cents(Z, Y, S, r)

        if all(
            k in cr
            for k in ["R", "C1", "C2"]
        ):
            X.append(
                np.r_[cr["R"], cr["C2"]]
            )
            T.append(cr["C1"])

    X = np.asarray(X)
    T = np.asarray(T)

    # Ridge baseline
    ridge = Ridge(alpha=1.0).fit(X, T)

    pred_r = ridge.predict(
        np.r_[cs["R"], cs["C2"]][None]
    )[0]

    # Population centroids
    pop1 = T.mean(0)

    pop0 = np.mean([
        subj_cents(Z, Y, S, r)["C0"]
        for r in tr_subjects
    ], axis=0)

    pop2 = np.mean([
        subj_cents(Z, Y, S, r)["C2"]
        for r in tr_subjects
    ], axis=0)

    # Translation baseline
    pred_trans = (
        pop1
        + (cs["C2"] - pop2)
    )

    # Midpoint baseline
    pred_mid = (
        cs["R"]
        + 0.5 * (cs["C2"] - cs["R"])
    )

    # Subject's actual 0→2 trajectory length
    d02 = np.linalg.norm(
        cs["C2"] - cs["C0"]
    )

    rows.append({
        "subject": s,
        "d02": d02,

        "population_error": np.linalg.norm(
            pop1 - cs["C1"]
        ),

        "midpoint_error": np.linalg.norm(
            pred_mid - cs["C1"]
        ),

        "translation_error": np.linalg.norm(
            pred_trans - cs["C1"]
        ),

        "ridge_error": np.linalg.norm(
            pred_r - cs["C1"]
        )
    })


st2 = pd.DataFrame(rows)


# Normalize all newly computed errors by subject-specific d02
for c in [
    "population_error",
    "midpoint_error",
    "translation_error",
    "ridge_error"
]:
    st2[c + "_normalized"] = (
        st2[c] / st2["d02"]
    )


# Load the authoritative LONG-format Stage-2 results
STAGE2_LONG = OUT / "stage2_calibration_comparison_64D.csv"
stage2_long = pd.read_csv(STAGE2_LONG)


# Add existing frozen-geometric error
fg = (
    stage2_long[
        stage2_long["method"] == "frozen_geometric"
    ][
        ["subject", "error_64D"]
    ]
    .rename(columns={
        "error_64D": "frozen_geometric_error"
    })
)


st2 = st2.merge(
    fg,
    on="subject",
    how="left"
)


# Normalize frozen-geometric error by d02
st2["frozen_geometric_normalized"] = (
    st2["frozen_geometric_error"]
    / st2["d02"]
)


display(st2)


st2.to_csv(
    EXP / "stage2_new_baselines_normalized.csv",
    index=False
)

,subject,d02,population_error,midpoint_error,translation_error,ridge_error,population_error_normalized,midpoint_error_normalized,translation_error_normalized,ridge_error_normalized,frozen_geometric_error,frozen_geometric_normalized
0,1,0.021680,0.054337,0.084958,0.082463,0.050368,2.506351,3.918773,3.803649,2.323267,0.056798,2.619850
1,2,0.088463,0.071156,0.026146,0.033822,0.071014,0.804367,0.295560,0.382329,0.802757,0.024916,0.281657
2,3,0.044216,0.139185,0.065468,0.041302,0.134789,3.147883,1.480650,0.934097,3.048456,0.036172,0.818088
3,4,0.067354,0.027844,0.086676,0.044073,0.027277,0.413401,1.286872,0.654351,0.404976,0.042476,0.630636
4,5,0.059927,0.051690,0.116989,0.056535,0.053905,0.862549,1.952176,0.943393,0.899505,0.077678,1.296205
5,6,0.059674,0.037841,0.036955,0.037671,0.037527,0.634132,0.619277,0.631273,0.628871,0.019430,0.325609
6,7,0.036039,0.093353,0.084817,0.045927,0.091693,2.590320,2.353459,1.274368,2.544254,0.045306,1.257117
7,8,0.045485,0.113199,0.050302,0.051728,0.110794,2.488682,1.105884,1.137250,2.435814,0.027177,0.597498
8,9,0.064458,0.056985,0.079453,0.015622,0.057810,0.884066,1.232634,0.242351,0.896859,0.029781,0.462019
9,10,0.069501,0.026162,0.015786,0.019033,0.026271,0.376421,0.227135,0.273855,0.377989,0.024018,0.345582


In [16]:
summary = pd.DataFrame({
    "method": [
        "Population",
        "Midpoint",
        "Translation",
        "Ridge",
        "Frozen geometric"
    ],
    "error_mean": [
        st2.population_error.mean(),
        st2.midpoint_error.mean(),
        st2.translation_error.mean(),
        st2.ridge_error.mean(),
        st2.frozen_geometric_error.mean()
    ],
    "error_sd": [
        st2.population_error.std(ddof=1),
        st2.midpoint_error.std(ddof=1),
        st2.translation_error.std(ddof=1),
        st2.ridge_error.std(ddof=1),
        st2.frozen_geometric_error.std(ddof=1)
    ],
    "normalized_mean": [
        st2.population_error_normalized.mean(),
        st2.midpoint_error_normalized.mean(),
        st2.translation_error_normalized.mean(),
        st2.ridge_error_normalized.mean(),
        st2.frozen_geometric_normalized.mean()
    ],
    "normalized_sd": [
        st2.population_error_normalized.std(ddof=1),
        st2.midpoint_error_normalized.std(ddof=1),
        st2.translation_error_normalized.std(ddof=1),
        st2.ridge_error_normalized.std(ddof=1),
        st2.frozen_geometric_normalized.std(ddof=1)
    ]
})

display(summary)

summary.to_csv(
    EXP / "stage2_baseline_summary.csv",
    index=False
)

,method,error_mean,error_sd,normalized_mean,normalized_sd
0,Population,0.064860,0.031883,1.591830,1.223564
1,Midpoint,0.052229,0.025477,1.201041,0.787674
2,Translation,0.041646,0.017665,1.028911,0.832316
3,Ridge,0.063370,0.030721,1.544754,1.163709
4,Frozen geometric,0.034542,0.016811,0.770288,0.523137


## 11. Band Weight Info

In [17]:
import torch
import numpy as np
import pandas as pd

bands = ["delta", "theta", "alpha", "beta"]

weight_rows = []

for s in SUBJECTS:

    p = LOSO / f"subject_{s:02d}" / "fignn_best.pth"

    ckpt = torch.load(
        p,
        map_location="cpu",
        weights_only=False
    )

    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        state = ckpt["model_state_dict"]
    else:
        state = ckpt

    # Find the BandWeightLayer parameter robustly
    candidates = [
        k for k in state.keys()
        if k.endswith("raw_weights")
    ]

    if len(candidates) != 1:
        print(f"Subject {s}: found keys =", candidates)
        raise RuntimeError(
            f"Could not uniquely identify raw_weights for subject {s}"
        )

    raw = state[candidates[0]].float()

    weights = torch.softmax(raw, dim=0).numpy()

    weight_rows.append({
        "subject": s,
        "delta": weights[0],
        "theta": weights[1],
        "alpha": weights[2],
        "beta": weights[3]
    })


band_weights = pd.DataFrame(weight_rows)

display(band_weights)

,subject,delta,theta,alpha,beta
0,1,0.236590,0.184988,0.218941,0.359481
1,2,0.229834,0.203581,0.243147,0.323438
2,3,0.202076,0.191467,0.236684,0.369772
3,4,0.213133,0.200441,0.233958,0.352468
4,5,0.229596,0.175564,0.248113,0.346727
5,6,0.225826,0.168821,0.244044,0.361309
6,7,0.203833,0.193011,0.300459,0.302697
7,8,0.215211,0.187392,0.238202,0.359195
8,9,0.216267,0.166751,0.260154,0.356828
9,10,0.232526,0.180467,0.242025,0.344982


In [19]:
band_summary = pd.DataFrame({
    "band": bands,
    "mean": [
        band_weights[b].mean()
        for b in bands
    ],
    "sd": [
        band_weights[b].std(ddof=1)
        for b in bands
    ],
    "min": [
        band_weights[b].min()
        for b in bands
    ],
    "max": [
        band_weights[b].max()
        for b in bands
    ]
})

display(band_summary)

band_weights.to_csv(
    EXP / "loso_band_weights_all29.csv",
    index=False
)

band_summary.to_csv(
    EXP / "loso_band_weights_summary.csv",
    index=False
)

print("\n29-fold band weights:")
for _, r in band_summary.iterrows():
    print(
        f"{r['band']:>5}: "
        f"{r['mean']:.4f} ± {r['sd']:.4f} "
        f"[{r['min']:.4f}, {r['max']:.4f}]"
    )

,band,mean,sd,min,max
0,delta,0.213044,0.014915,0.172031,0.236590
1,theta,0.195799,0.018439,0.166751,0.244059
2,alpha,0.248702,0.026063,0.206048,0.339445
3,beta,0.342455,0.026780,0.260995,0.387886



29-fold band weights:
delta: 0.2130 ± 0.0149 [0.1720, 0.2366]
theta: 0.1958 ± 0.0184 [0.1668, 0.2441]
alpha: 0.2487 ± 0.0261 [0.2060, 0.3394]
 beta: 0.3425 ± 0.0268 [0.2610, 0.3879]


## 12. Stage-2 improvement vs direction cosine

In [23]:
from scipy.stats import pearsonr, spearmanr

# =============================================================
# Stage 2 improvement vs Stage 1 trajectory direction
# =============================================================

# Stage-2 long-format results
stage2_long = pd.read_csv(
    OUT / "stage2_calibration_comparison_64D.csv"
)

population = (
    stage2_long[
        stage2_long["method"] == "population_only"
    ][["subject", "error_64D"]]
    .rename(columns={"error_64D": "population_error"})
)

frozen = (
    stage2_long[
        stage2_long["method"] == "frozen_geometric"
    ][["subject", "error_64D"]]
    .rename(columns={"error_64D": "frozen_error"})
)

corr_df = (
    population
    .merge(frozen, on="subject")
    .merge(
        stage1[["subject", "direction_cosine"]],
        on="subject"
    )
)

# Positive = frozen geometric improved over population-only
corr_df["improvement"] = (
    corr_df["population_error"]
    - corr_df["frozen_error"]
)

corr_df["improved"] = corr_df["improvement"] > 0


# =============================================================
# Correlation across all 29 subjects
# =============================================================

pearson_all = pearsonr(
    corr_df["direction_cosine"],
    corr_df["improvement"]
)

spearman_all = spearmanr(
    corr_df["direction_cosine"],
    corr_df["improvement"]
)


# =============================================================
# Correlation among the 7 non-improved subjects
# =============================================================

non_improved = corr_df[
    ~corr_df["improved"]
].copy()

pearson_non = pearsonr(
    non_improved["direction_cosine"],
    non_improved["improvement"]
)

spearman_non = spearmanr(
    non_improved["direction_cosine"],
    non_improved["improvement"]
)


# =============================================================
# Print results
# =============================================================

print("=" * 70)
print("STAGE 2 IMPROVEMENT vs STAGE 1 DIRECTION COSINE")
print("=" * 70)

print(
    f"Improved: "
    f"{corr_df['improved'].sum()}/29"
)

print(
    f"Non-improved: "
    f"{(~corr_df['improved']).sum()}/29"
)

print("\nAll 29 subjects:")
print(
    f"Pearson  r = {pearson_all.statistic:.4f}, "
    f"p = {pearson_all.pvalue:.4g}"
)
print(
    f"Spearman rho = {spearman_all.statistic:.4f}, "
    f"p = {spearman_all.pvalue:.4g}"
)

print("\nSeven non-improved subjects:")
print(
    f"Pearson  r = {pearson_non.statistic:.4f}, "
    f"p = {pearson_non.pvalue:.4g}"
)
print(
    f"Spearman rho = {spearman_non.statistic:.4f}, "
    f"p = {spearman_non.pvalue:.4g}"
)

print("\nNon-improved subjects:")
display(
    non_improved[
        [
            "subject",
            "direction_cosine",
            "population_error",
            "frozen_error",
            "improvement"
        ]
    ].sort_values("subject")
)


# =============================================================
# Save
# =============================================================

corr_df.to_csv(
    EXP / "stage2_improvement_vs_direction_cosine.csv",
    index=False
)


# =============================================================
# FINAL AUDIT
# =============================================================

audit = {
    "n_subjects": len(stage1),
    "primary_pass": int(stage1.primary_pass.sum()),
    "distance_order_pass": int(stage1.distance_order_pass.sum()),
    "mean_d01_d02": float(stage1.d01_d02.mean()),
    "mean_path_efficiency": float(stage1.path_efficiency.mean()),
    "mean_procrustes": float(
        stage1.procrustes_similarity.mean()
    ),
    "mean_perpendicular_distance": float(
        stage1.perpendicular_distance.mean()
    ),
    "stage2_improved": int(
        corr_df["improved"].sum()
    ),
    "stage2_non_improved": int(
        (~corr_df["improved"]).sum()
    ),
    "pearson_all_r": float(
        pearson_all.statistic
    ),
    "pearson_all_p": float(
        pearson_all.pvalue
    ),
    "spearman_all_rho": float(
        spearman_all.statistic
    ),
    "spearman_all_p": float(
        spearman_all.pvalue
    ),
    "pearson_non_improved_r": float(
        pearson_non.statistic
    ),
    "pearson_non_improved_p": float(
        pearson_non.pvalue
    ),
    "spearman_non_improved_rho": float(
        spearman_non.statistic
    ),
    "spearman_non_improved_p": float(
        spearman_non.pvalue
    )
}

print("\n" + "=" * 70)
print("FINAL AUDIT")
print("=" * 70)

print(json.dumps(audit, indent=2))

assert audit["n_subjects"] == 29
assert audit["primary_pass"] == 23
assert audit["distance_order_pass"] == 27
assert audit["stage2_improved"] == 22
assert audit["stage2_non_improved"] == 7

(EXP / "final_audit.json").write_text(
    json.dumps(audit, indent=2)
)

print("\nAll final exports written to:")
print(EXP)

STAGE 2 IMPROVEMENT vs STAGE 1 DIRECTION COSINE
Improved: 22/29
Non-improved: 7/29

All 29 subjects:
Pearson  r = -0.0060, p = 0.9753
Spearman rho = -0.3498, p = 0.0629

Seven non-improved subjects:
Pearson  r = -0.5664, p = 0.1849
Spearman rho = -0.5000, p = 0.2532

Non-improved subjects:


,subject,direction_cosine,population_error,frozen_error,improvement
0,1,-0.834839,0.054337,0.056798,-0.002461
3,4,0.954822,0.027844,0.042476,-0.014632
4,5,0.942994,0.051690,0.077678,-0.025988
18,19,0.888955,0.053983,0.065714,-0.011730
21,22,0.974870,0.050470,0.066246,-0.015775
22,23,0.602872,0.035595,0.046839,-0.011244
26,27,0.962827,0.032357,0.036350,-0.003993



FINAL AUDIT
{
  "n_subjects": 29,
  "primary_pass": 23,
  "distance_order_pass": 27,
  "mean_d01_d02": 0.47715064068965524,
  "mean_path_efficiency": 0.8191989051724137,
  "mean_procrustes": 0.87746762919339,
  "mean_perpendicular_distance": 0.013467512582758618,
  "stage2_improved": 22,
  "stage2_non_improved": 7,
  "pearson_all_r": -0.006013899437075029,
  "pearson_all_p": 0.9753002626705634,
  "spearman_all_rho": -0.3497536945812807,
  "spearman_all_p": 0.0629047444505732,
  "pearson_non_improved_r": -0.5664253504619371,
  "pearson_non_improved_p": 0.1849261845307234,
  "spearman_non_improved_rho": -0.5,
  "spearman_non_improved_p": 0.2531699951003227
}

All final exports written to:
D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\outputs\final_non_gpu_validation


## 13. Threshold sensitivity verification

In [ ]:
for p in [TH/"threshold_summary.csv",TH/"threshold_results.csv",TH/"threshold_0.5_v4_check.csv"]:
    print("\\n",p.name,p.exists())
    if p.exists(): 
        display(pd.read_csv(p).head(20))

\n threshold_summary.csv True


,threshold,subjects,between_count,between_rate,mean_d01_d02,std_d01_d02,mean_d12_d02,std_d12_d02,mean_relative_position,std_relative_position,...,std_perpendicular_distance,mean_path_efficiency,std_path_efficiency,mean_angle,std_angle,mean_procrustes,std_procrustes,mean_direction_cosine,std_direction_cosine,between_rate_%
0,0.3,29,21,0.724138,0.628058,0.535069,0.900184,0.480973,0.318598,0.465085,...,0.012085,0.767363,0.217247,83.785517,38.088045,0.806646,0.218721,0.699435,0.446682,72.413793
1,0.4,29,23,0.793103,0.536103,0.392645,0.865381,0.386489,0.271577,0.343124,...,0.010628,0.803625,0.205214,75.428975,39.601466,0.847213,0.193901,0.766563,0.395966,79.310345
2,0.5,29,23,0.793103,0.477151,0.269485,0.823923,0.283942,0.270550,0.290557,...,0.008846,0.819199,0.170369,73.196998,39.671425,0.877468,0.154761,0.785395,0.403633,79.310345
3,0.6,29,23,0.793103,0.478270,0.247467,0.862629,0.346123,0.214036,0.361057,...,0.005877,0.800368,0.170305,78.062188,42.257198,0.878463,0.142283,0.767682,0.382217,79.310345
4,0.7,29,25,0.862069,0.548655,0.415302,0.880549,0.549644,0.200246,0.555329,...,0.004728,0.788329,0.178190,79.764054,40.089830,0.849583,0.150661,0.745407,0.367577,86.206897


\n threshold_results.csv True


,threshold,subject,d01,d02,d12,d01_d02,d12_d02,angle_deg,path_efficiency,d01_lt_d02,relative_1back_position,centrality,perpendicular_distance,oneback_between_x,procrustes_similarity,direction_cosine,checkpoint,n_training_subjects
0,0.3,1,0.010148,0.012563,0.013802,0.807773,1.098627,118.963020,0.524549,True,0.222758,0.445517,0.009755,True,0.721956,-0.141625,/lp-dev/24BCE1793/outputs/loso/subject_01/fign...,28
1,0.3,2,0.036216,0.182782,0.155488,0.198140,0.850678,45.307831,0.953454,True,0.157803,0.315606,0.021902,True,0.977746,0.898332,/lp-dev/24BCE1793/outputs/loso/subject_02/fign...,28
2,0.3,3,0.021589,0.046331,0.052413,0.465969,1.131293,118.122559,0.626071,True,-0.031349,0.000000,0.021540,False,0.831725,0.837136,/lp-dev/24BCE1793/outputs/loso/subject_03/fign...,28
3,0.3,4,0.043217,0.066045,0.028109,0.654354,0.425607,45.460278,0.925959,True,0.623519,0.752962,0.013110,True,0.801861,0.917611,/lp-dev/24BCE1793/outputs/loso/subject_04/fign...,28
4,0.3,5,0.057522,0.066552,0.019302,0.864309,0.290034,70.318802,0.866294,True,0.831455,0.337090,0.015709,True,0.615049,0.949180,/lp-dev/24BCE1793/outputs/loso/subject_05/fign...,28
5,0.3,6,0.045192,0.061421,0.025766,0.735764,0.419502,62.747837,0.865602,True,0.682683,0.634633,0.016854,True,0.745569,0.959697,/lp-dev/24BCE1793/outputs/loso/subject_06/fign...,28
6,0.3,7,0.006750,0.027793,0.024110,0.242859,0.867479,63.425045,0.900626,True,0.153230,0.306460,0.005237,True,0.974414,0.721066,/lp-dev/24BCE1793/outputs/loso/subject_07/fign...,28
7,0.3,8,0.031642,0.075783,0.047140,0.417530,0.622036,32.366241,0.961940,True,0.393701,0.787403,0.010537,True,0.972671,0.901933,/lp-dev/24BCE1793/outputs/loso/subject_08/fign...,28
8,0.3,9,0.011728,0.087286,0.088729,0.134368,1.016538,100.845879,0.868881,True,-0.007647,0.000000,0.011709,False,0.962395,0.976421,/lp-dev/24BCE1793/outputs/loso/subject_09/fign...,28
9,0.3,10,0.019361,0.082081,0.068589,0.235872,0.835632,51.372608,0.933267,True,0.178677,0.357354,0.012639,True,0.984649,0.955218,/lp-dev/24BCE1793/outputs/loso/subject_10/fign...,28


\n threshold_0.5_v4_check.csv True


,threshold,subject,d01,d02,d12,d01_d02,d12_d02,angle_deg,path_efficiency,d01_lt_d02,relative_1back_position,centrality,perpendicular_distance,oneback_between_x,procrustes_similarity,direction_cosine,checkpoint,n_training_subjects
0,0.5,1,0.012123,0.021680,0.014933,0.559199,0.688781,106.042252,0.801295,True,0.419142,0.838285,0.008025,True,0.903420,-0.834838,/lp-dev/24BCE1793/outputs/loso/subject_01/fign...,28
1,0.5,2,0.024683,0.088463,0.077595,0.279016,0.877146,108.185791,0.864931,True,0.154232,0.308465,0.020569,True,0.962859,0.875531,/lp-dev/24BCE1793/outputs/loso/subject_02/fign...,28
2,0.5,3,0.017334,0.044216,0.033566,0.392029,0.759139,116.978569,0.868683,True,0.288697,0.577394,0.011727,True,0.965416,0.854275,/lp-dev/24BCE1793/outputs/loso/subject_03/fign...,28
3,0.5,4,0.043441,0.067354,0.028198,0.644963,0.418652,139.192261,0.940189,True,0.620354,0.759292,0.011885,True,0.809539,0.954823,/lp-dev/24BCE1793/outputs/loso/subject_04/fign...,28
4,0.5,5,0.032191,0.059927,0.033896,0.537167,0.565617,130.114899,0.906796,True,0.484313,0.968626,0.013924,True,0.912726,0.942994,/lp-dev/24BCE1793/outputs/loso/subject_05/fign...,28
5,0.5,6,0.032459,0.059674,0.035685,0.543945,0.598005,122.183678,0.875695,True,0.469133,0.938266,0.016428,True,0.907934,0.952020,/lp-dev/24BCE1793/outputs/loso/subject_06/fign...,28
6,0.5,7,0.011828,0.036039,0.025727,0.328198,0.713852,144.759933,0.959646,True,0.299065,0.598129,0.004872,True,0.998408,0.862118,/lp-dev/24BCE1793/outputs/loso/subject_07/fign...,28
7,0.5,8,0.020241,0.045485,0.026678,0.444999,0.586529,151.318634,0.969436,True,0.427004,0.854008,0.005698,True,0.964522,0.765699,/lp-dev/24BCE1793/outputs/loso/subject_08/fign...,28
8,0.5,9,0.010815,0.064458,0.065168,0.167778,1.011016,81.472893,0.848324,True,0.002998,0.005995,0.010813,True,0.943956,0.964994,/lp-dev/24BCE1793/outputs/loso/subject_09/fign...,28
9,0.5,10,0.012909,0.069501,0.061948,0.185734,0.891323,121.104790,0.928456,True,0.120020,0.240040,0.009852,True,0.974131,0.945760,/lp-dev/24BCE1793/outputs/loso/subject_10/fign...,28


## 14. Edge density at threshold 0.5

In [ ]:
possible = 62 * 61 // 2
counts = []

for g in all_graphs:

    ei = g.edge_index.numpy()

    pairs = {
        tuple(sorted((int(a), int(b))))
        for a, b in zip(ei[0], ei[1])
        if a != b
    }

    counts.append(len(pairs))


counts = np.asarray(counts)

density = counts / possible

print("possible:", possible)
print("mean edges:", counts.mean())
print("sd edges:", counts.std(ddof=1))
print("density:", density.mean())
print("sd density:", density.std(ddof=1))


pd.DataFrame({
    "n_edges": counts,
    "density": density
}).to_csv(
    EXP / "edge_density_threshold_05.csv",
    index=False
)

possible: 1891
mean edges: 558.1646425215065
sd edges: 145.9443973175378
density: 0.2951690335914895
sd density: 0.07717842269568367


## 15. REST statistical test

In [25]:
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests

rest_rows = []

for s in SUBJECTS:

    a = np.load(cache_path(s))

    Z = a["Z"]
    Y = a["Y"]
    S = a["S"]

    # Held-out subject only
    mask = S == s

    z = Z[mask]
    y = Y[mask]

    # Subject centroids
    R = z[y == 3].mean(axis=0)
    C0 = z[y == 0].mean(axis=0)
    C1 = z[y == 1].mean(axis=0)
    C2 = z[y == 2].mean(axis=0)

    rest_rows.append({
        "subject": s,
        "REST_0back": np.linalg.norm(C0 - R),
        "REST_1back": np.linalg.norm(C1 - R),
        "REST_2back": np.linalg.norm(C2 - R)
    })

rest_dist = pd.DataFrame(rest_rows)

# -------------------------------------------------------------
# Friedman omnibus test
# -------------------------------------------------------------

friedman_stat, friedman_p = friedmanchisquare(
    rest_dist["REST_0back"],
    rest_dist["REST_1back"],
    rest_dist["REST_2back"]
)

print("=" * 70)
print("REST → N-BACK DISTANCE ANALYSIS")
print("=" * 70)

print("\nMean ± SD:")
for c in [
    "REST_0back",
    "REST_1back",
    "REST_2back"
]:
    print(
        f"{c}: "
        f"{rest_dist[c].mean():.4f} ± "
        f"{rest_dist[c].std(ddof=1):.4f}"
    )

print(
    f"\nFriedman χ² = {friedman_stat:.4f}, "
    f"p = {friedman_p:.4g}"
)


# -------------------------------------------------------------
# Pairwise Wilcoxon tests
# -------------------------------------------------------------

pairs = [
    ("REST_0back", "REST_1back"),
    ("REST_0back", "REST_2back"),
    ("REST_1back", "REST_2back")
]

pair_results = []

for a_col, b_col in pairs:

    stat, p = wilcoxon(
        rest_dist[a_col],
        rest_dist[b_col],
        alternative="less"
    )

    pair_results.append({
        "comparison": f"{a_col} < {b_col}",
        "W": stat,
        "raw_p": p
    })

pair_results = pd.DataFrame(pair_results)

pair_results["holm_p"] = multipletests(
    pair_results["raw_p"],
    method="holm"
)[1]

print("\nPairwise Wilcoxon tests:")
display(pair_results)


# -------------------------------------------------------------
# Save
# -------------------------------------------------------------

rest_dist.to_csv(
    EXP / "rest_nback_distances.csv",
    index=False
)

pair_results.to_csv(
    EXP / "rest_nback_statistics.csv",
    index=False
)

REST → N-BACK DISTANCE ANALYSIS

Mean ± SD:
REST_0back: 0.1136 ± 0.0493
REST_1back: 0.1245 ± 0.0537
REST_2back: 0.1561 ± 0.0565

Friedman χ² = 37.4483, p = 7.383e-09

Pairwise Wilcoxon tests:


,comparison,W,raw_p,holm_p
0,REST_0back < REST_1back,71.0,4.873462e-04,4.873462e-04
1,REST_0back < REST_2back,5.0,1.862645e-08,5.587935e-08
2,REST_1back < REST_2back,7.0,3.539026e-08,7.078052e-08


## 16. COG-BCI Session Metadata Audit

This section audits the dataset-level session metadata stored in
`notebook.mat`, including session interruption flags, experimental block
order, and investigator notes.

The metadata audit is descriptive only. It is not used to modify the
EEG graphs, labels, embeddings, or statistical analyses.

In [30]:
# =============================================================
# COG-BCI session metadata audit
# =============================================================

from scipy.io import loadmat

assert NOTEBOOK_MAT.exists(), f"Missing: {NOTEBOOK_MAT}"

nb_mat = loadmat(
    NOTEBOOK_MAT,
    squeeze_me=True,
    struct_as_record=False
)

assert "notebook" in nb_mat, "Expected variable 'notebook' not found."

notebook = nb_mat["notebook"]

print("Notebook fields:")
print(notebook._fieldnames)

Notebook fields:
['SBJ_1', 'SBJ_2', 'SBJ_3', 'SBJ_4', 'SBJ_5', 'SBJ_6', 'SBJ_7', 'SBJ_8', 'SBJ_9', 'SBJ_10', 'SBJ_11', 'SBJ_12', 'SBJ_13', 'SBJ_14', 'SBJ_15', 'SBJ_16', 'SBJ_17', 'SBJ_18', 'SBJ_19', 'SBJ_20', 'SBJ_21', 'SBJ_22', 'SBJ_23', 'SBJ_24', 'SBJ_25', 'SBJ_26', 'SBJ_27', 'SBJ_28', 'SBJ_29']


In [31]:
# =============================================================
# Extract subject/session metadata
# =============================================================

metadata_rows = []

for s in range(1, 30):

    subject_struct = getattr(
        notebook,
        f"SBJ_{s}"
    )

    for sess in range(1, 4):

        session_struct = getattr(
            subject_struct,
            f"SESS_{sess}"
        )

        interrupted = int(
            getattr(
                session_struct,
                "interrupted",
                0
            )
        )

        order = getattr(
            session_struct,
            "Order",
            None
        )

        if order is not None:
            order = np.asarray(order).flatten().astype(int).tolist()
            order_string = "-".join(map(str, order))
        else:
            order_string = ""

        notes = getattr(
            session_struct,
            "Notes",
            ""
        )

        if isinstance(notes, np.ndarray):
            notes = str(notes.tolist())

        notes = str(notes).strip()

        metadata_rows.append({
            "subject": s,
            "session": sess,
            "interrupted": interrupted,
            "order": order_string,
            "notes": notes
        })


session_metadata = pd.DataFrame(
    metadata_rows
)

display(session_metadata)

print(
    "Total subject-session records:",
    len(session_metadata)
)

print(
    "Interrupted sessions:",
    session_metadata["interrupted"].sum()
)

print(
    "Uninterrupted sessions:",
    (session_metadata["interrupted"] == 0).sum()
)

,subject,session,interrupted,order,notes
0,1,1,0,4-6-3-1-5-7-2-8,
1,1,2,0,8-6-3-1-7-2-5-4,
2,1,3,0,3-5-2-6-8-1-4-7,
3,2,1,1,5-7-3-6-4-2-8-1,the Two Back (position 5) had to be recorded l...
4,2,2,0,6-4-3-8-5-1-7-2,
...,...,...,...,...,...
82,28,2,0,5-1-2-3-8-4-6-7,
83,28,3,0,6-5-3-4-8-7-1-2,
84,29,1,0,4-6-2-1-8-5-7-3,
85,29,2,0,1-2-5-3-7-6-8-4,


Total subject-session records: 87
Interrupted sessions: 8
Uninterrupted sessions: 79


In [32]:
session_metadata.to_csv(
    METADATA_OUT / "cog_bci_session_metadata.csv",
    index=False
)

print(
    "Saved:",
    METADATA_OUT / "cog_bci_session_metadata.csv"
)

Saved: D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\outputs\final_non_gpu_validation\dataset_metadata\cog_bci_session_metadata.csv


In [34]:
# =============================================================
# Cross-reference Stage-1 geometric failures
# =============================================================

FAILED_SUBJECTS = (
    stage1.loc[
        ~stage1["primary_pass"],
        "subject"
    ]
    .astype(int)
    .tolist()
)

print("Stage-1 geometric failures:")
print(FAILED_SUBJECTS)

failure_metadata = session_metadata[
    session_metadata["subject"].isin(FAILED_SUBJECTS)
].copy()

display(
    failure_metadata[
        [
            "subject",
            "session",
            "interrupted",
            "order",
            "notes"
        ]
    ]
)

Stage-1 geometric failures:
[12, 14, 19, 22, 25, 29]


,subject,session,interrupted,order,notes
33,12,1,0,6-2-7-4-1-5-3-8,
34,12,2,1,1-3-6-2-7-4-5-8,Mistook what task the participant was doing ad...
35,12,3,0,1-8-6-5-4-3-2-7,
39,14,1,0,2-5-6-8-7-1-3-4,
40,14,2,0,2-4-7-6-3-8-1-5,
41,14,3,0,6-3-7-8-5-1-2-4,
54,19,1,0,5-2-3-6-4-1-7-8,
55,19,2,0,1-4-2-5-3-8-7-6,
56,19,3,0,4-1-7-8-5-2-3-6,
63,22,1,0,3-2-1-5-4-6-8-7,


In [35]:
failure_metadata.to_csv(
    METADATA_OUT / "stage1_failure_session_metadata.csv",
    index=False
)

In [36]:
# =============================================================
# Subject-level failure metadata summary
# =============================================================

failure_summary = []

for s in FAILED_SUBJECTS:

    sub = session_metadata[
        session_metadata["subject"] == s
    ]

    interrupted_sessions = sub.loc[
        sub["interrupted"] == 1,
        "session"
    ].astype(int).tolist()

    notes = sub.loc[
        sub["notes"].str.len() > 0,
        ["session", "notes"]
    ]

    failure_summary.append({
        "subject": s,
        "interrupted_sessions": ",".join(
            map(str, interrupted_sessions)
        ),
        "n_interrupted": len(interrupted_sessions),
        "documented_notes": " | ".join(
            f"S{int(r.session)}: {r.notes}"
            for _, r in notes.iterrows()
        )
    })

failure_summary = pd.DataFrame(
    failure_summary
)

display(failure_summary)

failure_summary.to_csv(
    METADATA_OUT / "stage1_failure_subject_summary.csv",
    index=False
)

,subject,interrupted_sessions,n_interrupted,documented_notes
0,12,2,1,S2: Mistook what task the participant was doin...
1,14,,0,
2,19,,0,
3,22,3,1,S3: []
4,25,,0,
5,29,3,1,S3: []


In [37]:
# =============================================================
# All documented session notes
# =============================================================

documented_notes = session_metadata[
    session_metadata["notes"].str.len() > 0
].copy()

print(
    "Sessions with documented notes:",
    len(documented_notes)
)

display(documented_notes)

documented_notes.to_csv(
    METADATA_OUT / "all_documented_session_notes.csv",
    index=False
)

Sessions with documented notes: 8


,subject,session,interrupted,order,notes
3,2,1,1,5-7-3-6-4-2-8-1,the Two Back (position 5) had to be recorded l...
13,5,2,1,2-1-6-8-7-4-5-3,Two pack had to be redone (11:34 new recodring...
21,8,1,1,4-1-8-3-5-6-7-2,Crash after Oneback (position 1) RSME was reco...
24,9,1,1,3-6-8-4-7-5-2-1,redid the two back conditiion after having mis...
34,12,2,1,1-3-6-2-7-4-5-8,Mistook what task the participant was doing ad...
46,16,2,1,5-6-1-2-7-3-4-8,[]
65,22,3,1,2-5-3-4-7-1-8-6,[]
86,29,3,1,7-4-8-1-6-5-3-2,[]


## 17. COG-BCI Trigger Semantics Audit

This section records the documented event-code semantics used to define
task epochs. The trigger definitions are treated as acquisition metadata
and are not inferred from the processed graph labels.

In [39]:
# =============================================================
# COG-BCI trigger semantics audit
# =============================================================

trigger_df = pd.read_csv(
    TRIGGER_LIST
)

trigger_df.columns = [
    "code",
    "description"
]

trigger_df["code"] = (
    trigger_df["code"]
    .astype(str)
    .str.strip()
)

trigger_df["description"] = (
    trigger_df["description"]
    .astype(str)
    .str.strip()
)

# Authoritative N-back trigger mapping used by preprocessing
EXPECTED_TRIGGERS = {
    "6021": "ZEROBACK Normal Trial Onset",
    "6121": "ONEBACK Normal Trial Onset",
    "6221": "TWOBACK Normal Trial Onset",

    "6022": "ZEROBACK Hit Trial Onset",
    "6122": "ONEBACK Hit Trial Onset",
    "6222": "TWOBACK Hit Trial Onset",

    "6023": "ZEROBACK Conflict Trial Onset",
    "6123": "ONEBACK Conflict Trial Onset",
    "6223": "TWOBACK Conflict Trial Onset",

    "6132": "ONEBACK Correct Response",
    "6232": "TWOBACK Correct Response",
}

for code, expected_description in EXPECTED_TRIGGERS.items():

    match = trigger_df[
        trigger_df["code"] == code
    ]

    assert len(match) == 1, (
        f"Expected exactly one entry for trigger {code}"
    )

    actual = match.iloc[0]["description"]

    assert actual == expected_description, (
        f"Trigger {code}: expected "
        f"'{expected_description}', got '{actual}'"
    )

trigger_audit = trigger_df[
    trigger_df["code"].isin(EXPECTED_TRIGGERS.keys())
].copy()

display(trigger_audit)

trigger_audit.to_csv(
    METADATA_OUT / "cog_bci_nback_trigger_audit.csv",
    index=False
)

print("Trigger semantics audit PASSED.")

,code,description
3,6021,ZEROBACK Normal Trial Onset
4,6022,ZEROBACK Hit Trial Onset
5,6023,ZEROBACK Conflict Trial Onset
13,6121,ONEBACK Normal Trial Onset
14,6122,ONEBACK Hit Trial Onset
15,6123,ONEBACK Conflict Trial Onset
17,6132,ONEBACK Correct Response
23,6221,TWOBACK Normal Trial Onset
24,6222,TWOBACK Hit Trial Onset
25,6223,TWOBACK Conflict Trial Onset


Trigger semantics audit PASSED.


In [40]:
# =============================================================
# Trigger selection used by FIGNN preprocessing
# =============================================================

SELECTED_EPOCH_TRIGGERS = {
    0: "6021",
    1: "6121",
    2: "6221"
}

print(
    "Selected event-aligned N-back onset triggers:"
)

for label, code in SELECTED_EPOCH_TRIGGERS.items():

    description = trigger_df.loc[
        trigger_df["code"] == code,
        "description"
    ].iloc[0]

    print(
        f"  label {label}: {code} -> {description}"
    )

print(
    "\nThese are normal-trial onset events; "
    "hit/conflict/response triggers are not used "
    "as epoch-start events."
)

Selected event-aligned N-back onset triggers:
  label 0: 6021 -> ZEROBACK Normal Trial Onset
  label 1: 6121 -> ONEBACK Normal Trial Onset
  label 2: 6221 -> TWOBACK Normal Trial Onset

These are normal-trial onset events; hit/conflict/response triggers are not used as epoch-start events.


## 18. Graph-Count and Trigger-Consistency Audit

This section verifies that the processed graph dataset has the expected
subject-, session-, and workload-level counts implied by the event-aligned
N-back preprocessing and the retained REST graphs.

In [46]:
# ============================================================
# 18. Graph-count / provenance consistency audit
# ============================================================

import numpy as np
import pandas as pd
import torch
from collections import Counter

print("=" * 70)
print("18. GRAPH-COUNT / PROVENANCE CONSISTENCY AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# Extract metadata directly from PyG Data objects
# Avoids collisions with notebook variables/functions such as
# attr, label, subject, etc.
# ------------------------------------------------------------

graph_subjects = np.array([
    int(torch.as_tensor(g.subject_id).view(-1)[0])
    for g in all_graphs
])

graph_sessions = np.array([
    int(torch.as_tensor(g.session_id).view(-1)[0])
    for g in all_graphs
])

graph_labels = np.array([
    int(torch.as_tensor(g.y).view(-1)[0])
    for g in all_graphs
])

# ------------------------------------------------------------
# 1. Overall graph count
# ------------------------------------------------------------

print("\nTOTAL GRAPHS")
print("-" * 50)

print(f"Total: {len(all_graphs)}")

assert len(all_graphs) == 27666, \
    f"Expected 27666 graphs, got {len(all_graphs)}"

print("PASS: total graph count = 27,666")


# ------------------------------------------------------------
# 2. Overall label counts
# ------------------------------------------------------------

print("\nOVERALL LABEL COUNTS")
print("-" * 50)

label_counts = Counter(graph_labels)

expected_labels = {
    0: 8352,
    1: 8352,
    2: 8352,
    3: 2610,
}

for lab, expected in expected_labels.items():
    actual = label_counts.get(lab, 0)

    print(
        f"Label {lab}: "
        f"{actual} graphs "
        f"(expected {expected})"
    )

    assert actual == expected, \
        f"Label {lab}: expected {expected}, got {actual}"

print("PASS: overall label counts match")


# ------------------------------------------------------------
# 3. Per-subject counts
# ------------------------------------------------------------

print("\nPER-SUBJECT COUNTS")
print("-" * 50)

subject_rows = []

for s in range(1, 30):

    mask = graph_subjects == s

    counts = {
        lab: int(np.sum(mask & (graph_labels == lab)))
        for lab in [0, 1, 2, 3]
    }

    total = int(np.sum(mask))

    subject_rows.append({
        "subject": s,
        "label_0": counts[0],
        "label_1": counts[1],
        "label_2": counts[2],
        "label_3": counts[3],
        "total": total,
    })

subject_df = pd.DataFrame(subject_rows)

display(subject_df)

assert np.all(subject_df["label_0"] == 288)
assert np.all(subject_df["label_1"] == 288)
assert np.all(subject_df["label_2"] == 288)
assert np.all(subject_df["label_3"] == 90)
assert np.all(subject_df["total"] == 954)

print("PASS: every subject has 288/288/288/90 graphs")
print("PASS: every subject has 954 graphs total")


# ------------------------------------------------------------
# 4. Per-subject/session counts
# ------------------------------------------------------------

print("\nPER-SUBJECT / SESSION COUNTS")
print("-" * 50)

session_rows = []

for s in range(1, 30):
    for sess in [1, 2, 3]:

        mask = (
            (graph_subjects == s) &
            (graph_sessions == sess)
        )

        counts = {
            lab: int(np.sum(mask & (graph_labels == lab)))
            for lab in [0, 1, 2, 3]
        }

        total = int(np.sum(mask))

        session_rows.append({
            "subject": s,
            "session": sess,
            "label_0": counts[0],
            "label_1": counts[1],
            "label_2": counts[2],
            "label_3": counts[3],
            "total": total,
        })

session_df = pd.DataFrame(session_rows)

display(session_df)

assert np.all(session_df["label_0"] == 96)
assert np.all(session_df["label_1"] == 96)
assert np.all(session_df["label_2"] == 96)
assert np.all(session_df["label_3"] == 30)
assert np.all(session_df["total"] == 318)

print("PASS: every subject/session has 96/96/96/30 graphs")
print("PASS: every subject/session has 318 graphs")


# ------------------------------------------------------------
# 5. Save audit tables
# ------------------------------------------------------------

subject_df.to_csv(
    "cog_bci_graph_counts_by_subject.csv",
    index=False
)

session_df.to_csv(
    "cog_bci_graph_counts_by_subject_session.csv",
    index=False
)

print("\nSaved:")
print("  cog_bci_graph_counts_by_subject.csv")
print("  cog_bci_graph_counts_by_subject_session.csv")

print("\n" + "=" * 70)
print("SECTION 18 COMPLETE")
print("=" * 70)

18. GRAPH-COUNT / PROVENANCE CONSISTENCY AUDIT


AttributeError: 'GlobalStorage' object has no attribute 'subject_id'

## 19. MAT FILES

In [44]:
from scipy.io import loadmat
from pathlib import Path
import numpy as np

p = Path(
    r"D:\Portfolio Projects\QML EEG Paper\fignn_cni__project"
    r"\data\data\raw\sub-01\ses-S1\behavioral\0-Back.mat"
)

mat = loadmat(p, squeeze_me=True, struct_as_record=False)

print("FILE:", p)
print("\nVARIABLES:\n")

for k, v in mat.items():
    if k.startswith("__"):
        continue

    arr = np.asarray(v)

    print(f"{k:30s} | shape={str(arr.shape):15s} | dtype={arr.dtype}")

FILE: D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\data\raw\sub-01\ses-S1\behavioral\0-Back.mat

VARIABLES:

None                           | shape=(1,)            | dtype=[('s0', 'O'), ('s1', 'O'), ('s2', 'O'), ('arr', 'O')]


In [45]:
files = [
    r"D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\data\raw\sub-01\ses-S1\behavioral\0-Back.mat",
    r"D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\data\raw\sub-01\ses-S1\behavioral\1-Back.mat",
    r"D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\data\raw\sub-01\ses-S1\behavioral\2-Back.mat",
]

for f in files:
    mat = loadmat(f, squeeze_me=True, struct_as_record=False)

    print("\n" + "=" * 80)
    print(Path(f).name)

    for k, v in mat.items():
        if k.startswith("__"):
            continue

        arr = np.asarray(v)
        print(f"{k:30s} | shape={str(arr.shape):15s} | dtype={arr.dtype}")


0-Back.mat
None                           | shape=(1,)            | dtype=[('s0', 'O'), ('s1', 'O'), ('s2', 'O'), ('arr', 'O')]

1-Back.mat
None                           | shape=(1,)            | dtype=[('s0', 'O'), ('s1', 'O'), ('s2', 'O'), ('arr', 'O')]

2-Back.mat
None                           | shape=(1,)            | dtype=[('s0', 'O'), ('s1', 'O'), ('s2', 'O'), ('arr', 'O')]


In [48]:
from scipy.io import loadmat
import numpy as np

p = r"D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\data\raw\sub-01\ses-S1\behavioral\0-Back.mat"

mat = loadmat(p, squeeze_me=False, struct_as_record=True)

print("TOP LEVEL:")
for k, v in mat.items():
    if not k.startswith("__"):
        print(k, type(v), np.asarray(v).shape, np.asarray(v).dtype)

print("\n" + "="*70)

x = mat["None"]

print("STRUCT:")
print("shape:", x.shape)
print("dtype:", x.dtype)
print("fields:", x.dtype.names)

for field in x.dtype.names:
    v = x[field]

    print("\n" + "-"*50)
    print("FIELD:", field)
    print("type:", type(v))
    print("shape:", np.asarray(v).shape)
    print("dtype:", np.asarray(v).dtype)
    print("value:")
    print(v)

TOP LEVEL:
None <class 'scipy.io.matlab._mio5_params.MatlabOpaque'> (1,) [('s0', 'O'), ('s1', 'O'), ('s2', 'O'), ('arr', 'O')]

STRUCT:
shape: (1,)
dtype: [('s0', 'O'), ('s1', 'O'), ('s2', 'O'), ('arr', 'O')]
fields: ('s0', 's1', 's2', 'arr')

--------------------------------------------------
FIELD: s0
type: <class 'scipy.io.matlab._mio5_params.MatlabOpaque'>
shape: (1,)
dtype: object
value:
[b'nback']

--------------------------------------------------
FIELD: s1
type: <class 'scipy.io.matlab._mio5_params.MatlabOpaque'>
shape: (1,)
dtype: object
value:
[b'MCOS']

--------------------------------------------------
FIELD: s2
type: <class 'scipy.io.matlab._mio5_params.MatlabOpaque'>
shape: (1,)
dtype: object
value:
[b'table']

--------------------------------------------------
FIELD: arr
type: <class 'scipy.io.matlab._mio5_params.MatlabOpaque'>
shape: (1,)
dtype: object
value:
[array([[3707764736],
        [         2],
        [         1],
        [         1],
        [         1],
  

In [51]:
from scipy.io import loadmat

p = r"D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\data\raw\sub-01\ses-S1\behavioral\0-Back.mat"

mat = loadmat(
    p,
    squeeze_me=False,
    struct_as_record=False,
    simplify_cells=False
)

for k, v in mat.items():
    print("\nKEY:", repr(k))
    print("TYPE:", type(v))
    print("SHAPE:", getattr(v, "shape", None))
    print("VALUE:")
    print(repr(v))


KEY: '__header__'
TYPE: <class 'bytes'>
SHAPE: None
VALUE:
b'MATLAB 5.0 MAT-file, Platform: PCWIN64, Created on: Wed Jul  6 17:22:50 2022'

KEY: '__version__'
TYPE: <class 'str'>
SHAPE: None
VALUE:
'1.0'

KEY: '__globals__'
TYPE: <class 'list'>
SHAPE: None
VALUE:
[]

KEY: 'None'
TYPE: <class 'scipy.io.matlab._mio5_params.MatlabOpaque'>
SHAPE: (1,)
VALUE:
MatlabOpaque([(b'nback', b'MCOS', b'table', array([[3707764736],
                     [         2],
                     [         1],
                     [         1],
                     [         1],
                     [         1]], dtype=uint32))              ],
             dtype=[('s0', 'O'), ('s1', 'O'), ('s2', 'O'), ('arr', 'O')])

KEY: '__function_workspace__'
TYPE: <class 'numpy.ndarray'>
SHAPE: (1, 19328)
VALUE:
array([[ 0,  1, 73, ...,  0,  0,  0]], shape=(1, 19328), dtype=uint8)


In [53]:
import re

workspace = mat["__function_workspace__"].tobytes()

strings = re.findall(rb'[\x20-\x7e]{4,}', workspace)

print(f"Found {len(strings)} readable strings:\n")

for s in strings:
    print(s.decode("ascii", errors="replace"))

Found 40 readable strings:

MCOS
MCOS
FileWrapper__
data
ndims
nrows
rownames
nvars
varnames
props
table
block
hittrials
sess
miss
error
mistake
correct
condition
class
outlier
useVariableNamesOriginal
useDimensionNamesOriginal
CustomProps
VariableCustomProps
versionSavedFrom
minCompatibleVersion
incompatibilityMsg
VersionSavedFrom
Description
VariableNamesOriginal
DimensionNames
DimensionNamesOriginal
UserData
VariableDescriptions
VariableUnits
VariableContinuity
Variables
data
MCOS


In [54]:
for term in [
    b'hittrials',
    b'sess',
    b'miss',
    b'error',
    b'mistake',
    b'correct',
    b'condition',
    b'class',
    b'outlier',
]:
    pos = workspace.lower().find(term.lower())

    print(f"{term.decode():12s} -> {pos}")

hittrials    -> 16816
sess         -> 16940
miss         -> 16996
error        -> 17056
mistake      -> 17120
correct      -> 17184
condition    -> 17248
class        -> 17320
outlier      -> 17384


In [55]:
for term in [b'varnames', b'hittrials', b'Variables']:
    pos = workspace.find(term)

    print("\n" + "=" * 70)
    print(term, "at", pos)

    if pos >= 0:
        start = max(0, pos - 150)
        end = min(len(workspace), pos + 300)

        print(workspace[start:end])
        


b'varnames' at 320
b'\x00\x00\x08\x00\x00\x00\x0b\x00\x00\x00\x01\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x0e\x00\x00\x00P\x01\x00\x00\x06\x00\x00\x00\x08\x00\x00\x00\t\x00\x00\x00\x00\x00\x00\x00\x05\x00\x00\x00\x08\x00\x00\x00 \x01\x00\x00\x01\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x02\x00\x00\x00 \x01\x00\x00\x03\x00\x00\x00\x08\x00\x00\x00`\x00\x00\x00\x80\x00\x00\x00\x80\x00\x00\x00\xb0\x00\x00\x00\x10\x01\x00\x00 \x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00data\x00ndims\x00nrows\x00rownames\x00nvars\x00varnames\x00props\x00table\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x08\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x07\x00\x00\x00\x01\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x02\x00\x00\x00\x

In [56]:
# Show all readable strings with their byte positions
for s in strings:
    pos = workspace.find(s)
    print(f"{pos:6d}  {s.decode('ascii', errors='replace')}")

    72  MCOS
    72  MCOS
   128  FileWrapper__
   288  data
   293  ndims
   299  nrows
   305  rownames
   314  nvars
   320  varnames
   329  props
   335  table
 16640  block
 16816  hittrials
 16940  sess
 16996  miss
 17056  error
 17120  mistake
 17184  correct
 17248  condition
 17320  class
 17384  outlier
 17456  useVariableNamesOriginal
 17482  useDimensionNamesOriginal
 17508  CustomProps
 17534  VariableCustomProps
 17560  versionSavedFrom
 17586  minCompatibleVersion
 17612  incompatibilityMsg
 17638  VersionSavedFrom
 17664  Description
 17459  VariableNamesOriginal
 17485  DimensionNames
 17485  DimensionNamesOriginal
 17768  UserData
 17794  VariableDescriptions
 17820  VariableUnits
 17846  VariableContinuity
 18624  Variables
   288  data
    72  MCOS


In [57]:
for pos in [0, 3000, 6000, 9000, 12000, 15000]:
    print("\n" + "=" * 30, pos, "=" * 30)
    print(workspace[pos:pos+500])


============================== 0 ==============================
b'\x00\x01IM\x00\x00\x00\x00\x0e\x00\x00\x00\xe0J\x00\x00\x06\x00\x00\x00\x08\x00\x00\x00\x02\x00\x00\x00\x00\x00\x00\x00\x05\x00\x00\x00\x08\x00\x00\x00\x01\x00\x00\x00\x01\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x05\x00\x04\x00\x05\x00\x00\x00\x01\x00\x00\x00\x05\x00\x00\x00MCOS\x00\x00\x00\x00\x0e\x00\x00\x00\x98J\x00\x00\x06\x00\x00\x00\x08\x00\x00\x00\x11\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x01\x00\x04\x00MCOS\x01\x00\x00\x00\r\x00\x00\x00FileWrapper__\x00\x00\x00\x0e\x00\x00\x00XJ\x00\x00\x06\x00\x00\x00\x08\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x05\x00\x00\x00\x08\x00\x00\x00\x0b\x00\x00\x00\x01\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x0e\x00\x00\x00P\x01\x00\x00\x06\x00\x00\x00\x08\x00\x00\x00\t\x00\x00\x00\x00\x00\x00\x00\x05\x00\x00\x00\x08\x00\x00\x00 \x01\x00\x00\x01\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x02\x00\x00\x00 \x01\x00\x00\x03\x00\x00\x00\x08\x00\x00\x00`\x

In [58]:
for i in range(0, len(workspace), 256):
    chunk = workspace[i:i+256]

    printable = ''.join(
        chr(b) if 32 <= b < 127 else '.'
        for b in chunk
    )

    print(f"{i:05d}: {printable}")

00000: ..IM.........J..........................................................MCOS.........J..............................MCOS........FileWrapper__.......XJ..............................................P........................... ................... ...........
00256: `................... ...........data.ndims.nrows.rownames.nvars.varnames.props.table............................................................................................................................................................................
00512: .....................................=.........................................................................................................?.......?.......?.......?.......?.......?.......?.......?.......?.......?.......?.......?.......?.......?.......?
00768: .......?.......?.......?.......?.......?.......?.......?.......?.......?.......?.......?.......?.......?.......?.......?.......?.......?.......?.......?.......?.......?.......?.......?.......?.......?.

In [59]:
from pathlib import Path
import struct

p = Path(r"C:\Users")  # ignore this line; we'll locate TEMP below

tail = Path(__import__("tempfile").gettempdir()) / "sub12_tail.bin"
data = tail.read_bytes()

print("Tail size:", len(data))

# ZIP End of Central Directory signature
sig = b"PK\x05\x06"
pos = data.rfind(sig)

print("EOCD position:", pos)

if pos == -1:
    raise RuntimeError("EOCD not found in last 1 MB")

# EOCD structure:
# signature
# disk number
# central directory disk
# entries on disk
# total entries
# central directory size
# central directory offset
fields = struct.unpack_from("<4sHHHHIIH", data, pos)

_, disk, cd_disk, entries_disk, entries_total, cd_size, cd_offset, comment_len = fields

print("Entries:", entries_total)
print("Central directory size:", cd_size)
print("Central directory offset:", cd_offset)
print("Comment length:", comment_len)

Tail size: 1048576
EOCD position: 1048554
Entries: 113
Central directory size: 13409
Central directory offset: 1121345878
Comment length: 0


In [60]:
from pathlib import Path
import struct

tail = Path(__import__("tempfile").gettempdir()) / "sub12_tail.bin"
data = tail.read_bytes()

# EOCD
eocd = data.rfind(b"PK\x05\x06")
_, _, _, _, entries, cd_size, cd_offset, _ = struct.unpack_from(
    "<4sHHHHIIH", data, eocd
)

# Position of central directory inside our downloaded tail
tail_start = 1121359309 - len(data)
cd_pos = cd_offset - tail_start

print("Central directory in tail:", cd_pos)
print("Entries:", entries)
print()

pos = cd_pos

for i in range(entries):
    if data[pos:pos+4] != b"PK\x01\x02":
        raise RuntimeError(f"Bad central-directory signature at {pos}")

    # Central Directory File Header
    fields = struct.unpack_from(
        "<4s6H3I5H2I",
        data,
        pos
    )

    (
        sig, version_made, version_needed, flags, compression,
        mod_time, mod_date, crc32,
        compressed_size, uncompressed_size,
        filename_len, extra_len, comment_len,
        disk_start, internal_attr, external_attr,
        local_header_offset
    ) = fields

    name_start = pos + 46
    name = data[name_start:name_start + filename_len].decode(
        "utf-8", errors="replace"
    )

    if "behavioral" in name.lower() and name.lower().endswith(".mat"):
        print(
            f"{name}\n"
            f"  compressed:   {compressed_size:,} bytes\n"
            f"  uncompressed: {uncompressed_size:,} bytes\n"
            f"  local offset: {local_header_offset:,}\n"
            f"  compression:  {compression}\n"
        )

    pos += 46 + filename_len + extra_len + comment_len

Central directory in tail: 1035145
Entries: 113

sub-12/sub-12/ses-S1/behavioral/0-Back.mat
  compressed:   1,558 bytes
  uncompressed: 1,561 bytes
  local offset: 194
  compression:  8

sub-12/sub-12/ses-S1/behavioral/1-Back.mat
  compressed:   1,594 bytes
  uncompressed: 1,597 bytes
  local offset: 1,824
  compression:  8

sub-12/sub-12/ses-S1/behavioral/2-Back.mat
  compressed:   1,609 bytes
  uncompressed: 1,612 bytes
  local offset: 3,490
  compression:  8

sub-12/sub-12/ses-S1/behavioral/Flanker.mat
  compressed:   2,200 bytes
  uncompressed: 2,220 bytes
  local offset: 5,171
  compression:  8

sub-12/sub-12/ses-S1/behavioral/MATB_Diff.mat
  compressed:   10,825 bytes
  uncompressed: 10,825 bytes
  local offset: 7,444
  compression:  0

sub-12/sub-12/ses-S1/behavioral/MATB_Easy.mat
  compressed:   9,521 bytes
  uncompressed: 9,521 bytes
  local offset: 18,344
  compression:  0

sub-12/sub-12/ses-S1/behavioral/MATB_Med.mat
  compressed:   10,538 bytes
  uncompressed: 10,538 bytes


In [61]:
import os
import io
import struct
import zlib
import requests
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================

URL = "https://zenodo.org/records/7413650/files/sub-12.zip?download=1"

OUT_ROOT = Path(
    r"D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\data\raw"
)

SUBJECT = "sub-12"

# Exact entries we want
files = {
    "sub-12/ses-S1/behavioral/0-Back.mat": 194,
    "sub-12/ses-S1/behavioral/1-Back.mat": 1824,
    "sub-12/ses-S1/behavioral/2-Back.mat": 3490,

    "sub-12/ses-S2/behavioral/0-Back.mat": 376860147,
    "sub-12/ses-S2/behavioral/1-Back.mat": 376861754,
    "sub-12/ses-S2/behavioral/2-Back.mat": 376863450,

    "sub-12/ses-S3/behavioral/0-Back.mat": 749665742,
    "sub-12/ses-S3/behavioral/1-Back.mat": 749667375,
    "sub-12/ses-S3/behavioral/2-Back.mat": 749669065,
}


# ============================================================
# DOWNLOAD ONE FILE FROM ZIP USING RANGE REQUESTS
# ============================================================

def download_file(zip_offset, zip_name):
    # First fetch enough bytes to read the local ZIP header.
    r = requests.get(
        URL,
        headers={"Range": f"bytes={zip_offset}-{zip_offset + 4095}"},
        timeout=60,
    )
    r.raise_for_status()

    header = r.content

    if header[:4] != b"PK\x03\x04":
        raise RuntimeError(
            f"Invalid ZIP local header for {zip_name}: "
            f"{header[:20]!r}"
        )

    # Local file header:
    # signature       4
    # version         2
    # flags           2
    # compression     2
    # time            2
    # date            2
    # crc             4
    # compressed size 4
    # uncompressed    4
    # filename len    2
    # extra len       2

    (
        sig,
        version,
        flags,
        compression,
        mod_time,
        mod_date,
        crc32,
        compressed_size,
        uncompressed_size,
        filename_len,
        extra_len,
    ) = struct.unpack_from("<4s5H3I2H", header, 0)

    data_start = (
        zip_offset
        + 30
        + filename_len
        + extra_len
    )

    data_end = data_start + compressed_size - 1

    print(f"\n{zip_name}")
    print(f"  compression:     {compression}")
    print(f"  compressed size: {compressed_size:,}")
    print(f"  original size:   {uncompressed_size:,}")
    print(f"  data range:      {data_start:,} - {data_end:,}")

    # Download ONLY the compressed payload
    r = requests.get(
        URL,
        headers={"Range": f"bytes={data_start}-{data_end}"},
        timeout=60,
    )
    r.raise_for_status()

    compressed = r.content

    if len(compressed) != compressed_size:
        raise RuntimeError(
            f"Expected {compressed_size} bytes, "
            f"got {len(compressed)}"
        )

    # Compression method 0 = stored
    if compression == 0:
        raw = compressed

    # Compression method 8 = DEFLATE
    elif compression == 8:
        raw = zlib.decompress(compressed, -15)

    else:
        raise RuntimeError(
            f"Unsupported ZIP compression method: {compression}"
        )

    if len(raw) != uncompressed_size:
        raise RuntimeError(
            f"Size mismatch: expected {uncompressed_size}, "
            f"got {len(raw)}"
        )

    # Save to exact local dataset structure
    relative = Path(zip_name).relative_to(SUBJECT)
    output = OUT_ROOT / SUBJECT / relative

    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_bytes(raw)

    print(f"  SAVED: {output}")
    print(f"  size:  {len(raw):,} bytes")


# ============================================================
# RUN
# ============================================================

print("=" * 70)
print("COG-BCI S12 — BEHAVIORAL N-BACK EXTRACTION")
print("=" * 70)

total = 0

for name, offset in files.items():
    download_file(offset, name)
    total += 1

print("\n" + "=" * 70)
print(f"DONE — extracted {total} files")
print("=" * 70)

# Verify
for name in files:
    relative = Path(name).relative_to(SUBJECT)
    p = OUT_ROOT / SUBJECT / relative
    print(f"{p} -> {p.stat().st_size:,} bytes")

COG-BCI S12 — BEHAVIORAL N-BACK EXTRACTION

sub-12/ses-S1/behavioral/0-Back.mat
  compression:     8
  compressed size: 1,558
  original size:   1,561
  data range:      266 - 1,823
  SAVED: D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\data\raw\sub-12\ses-S1\behavioral\0-Back.mat
  size:  1,561 bytes

sub-12/ses-S1/behavioral/1-Back.mat
  compression:     8
  compressed size: 1,594
  original size:   1,597
  data range:      1,896 - 3,489
  SAVED: D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\data\raw\sub-12\ses-S1\behavioral\1-Back.mat
  size:  1,597 bytes

sub-12/ses-S1/behavioral/2-Back.mat
  compression:     8
  compressed size: 1,609
  original size:   1,612
  data range:      3,562 - 5,170
  SAVED: D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\data\raw\sub-12\ses-S1\behavioral\2-Back.mat
  size:  1,612 bytes

sub-12/ses-S2/behavioral/0-Back.mat
  compression:     0
  compressed size: 1,535
  original size:   1,535
  data range:      376,

In [62]:
from scipy.io import loadmat
from pathlib import Path

p = Path(
    r"D:\Portfolio Projects\QML EEG Paper\fignn_cni__project"
    r"\data\data\raw\sub-12\ses-S1\behavioral\0-Back.mat"
)

mat = loadmat(
    p,
    squeeze_me=False,
    struct_as_record=True
)

print(mat.keys())

obj = mat["None"]

print("\nTYPE:", type(obj))
print("SHAPE:", obj.shape)
print("DTYPE:", obj.dtype)

print("\nVALUE:")
print(obj)

dict_keys(['__header__', '__version__', '__globals__', 'None', '__function_workspace__'])

TYPE: <class 'scipy.io.matlab._mio5_params.MatlabOpaque'>
SHAPE: (1,)
DTYPE: [('s0', 'O'), ('s1', 'O'), ('s2', 'O'), ('arr', 'O')]

VALUE:
[(b'nback', b'MCOS', b'table', array([[3707764736],
        [         2],
        [         1],
        [         1],
        [         1],
        [         1]], dtype=uint32))              ]


## 15. Final audit/export

In [26]:
audit = {
    "n_subjects": len(stage1),
    "primary_pass": int(stage1.primary_pass.sum()),
    "distance_order_pass": int(stage1.distance_order_pass.sum()),
    "mean_d01_d02": float(stage1.d01_d02.mean()),
    "mean_path_efficiency": float(stage1.path_efficiency.mean()),
    "mean_procrustes": float(stage1.procrustes_similarity.mean()),
    "mean_perpendicular_distance": float(
        stage1.perpendicular_distance.mean()
    )
}

print(json.dumps(audit, indent=2))

assert audit["n_subjects"] == 29
assert audit["primary_pass"] == 23
assert audit["distance_order_pass"] == 27

(EXP / "final_audit.json").write_text(
    json.dumps(audit, indent=2)
)

print("All exports:", EXP)

{
  "n_subjects": 29,
  "primary_pass": 23,
  "distance_order_pass": 27,
  "mean_d01_d02": 0.47715064068965524,
  "mean_path_efficiency": 0.8191989051724137,
  "mean_procrustes": 0.87746762919339,
  "mean_perpendicular_distance": 0.013467512582758618
}
All exports: D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\outputs\final_non_gpu_validation
